# Comparing GEPA Experiments

This notebook uses the new DataFrame-based API for flexible experiment comparison.

In [ ]:
from dotenv import load_dotenv

from core.experiment.plots import (
    PlotConfig,
    Figure,
    Grid,
    BarPlot,
    ProgressionPlot,
    ScatterPlot,
    with_subsets,
    make_prompt_type_column,
    make_verbalization_float_column,
)

load_dotenv()

# =============================================================================
# CONFIGS
# =============================================================================

METRIC_COLUMNS = ["proxy_reward", "true_reward", "prompt_verbalizes"]

CONFIGS = {
    # MCQ - Claude prompter
    "mcq_claude": PlotConfig(
        paths=["logs/mcq/2025-11-19-11-41-43/"],
        quick_mode=False,
        figures=[
            Figure(
                name="Final scores",
                filter=lambda df: df.is_final & ~df.is_sanitized,
                layout=Grid(
                    groupby="hint_type",
                    cols_wrap=2,
                    inner=with_subsets(
                        BarPlot(x="suggest_hack", y=METRIC_COLUMNS, hue="column_name")
                    ),
                ),
            ),
            Figure(
                name="Training progression",
                filter=lambda df: ~df.is_sanitized,
                layout=Grid(
                    groupby="hint_type",
                    cols_wrap=2,
                    inner=with_subsets(
                        ProgressionPlot(
                            x="discovery_eval_counts",
                            y=METRIC_COLUMNS,
                            hue="column_name",
                        )
                    ),
                ),
            ),
        ],
    ),
    # Wordchain - Claude prompter
    "wordchain_claude": PlotConfig(
        paths=["logs/wordchain/2025-11-19-01-04-44/"],
        quick_mode=False,
        computed_columns={"prompt_verbalizes": make_verbalization_float_column},
        figures=[
            Figure(
                name="Final scores",
                filter=lambda df: df.is_final & ~df.is_sanitized,
                layout=BarPlot(x="suggest_hack", y=METRIC_COLUMNS, hue="column_name"),
            ),
            Figure(
                name="Training progression",
                # filter=lambda df: ~df.is_sanitized,
                layout=ProgressionPlot(
                    x="discovery_eval_counts",
                    y=METRIC_COLUMNS,
                    hue="column_name",
                    linestyle="is_sanitized",
                ),
            ),
            Figure(
                name="Training progression",
                # filter=lambda df: ~df.is_sanitized,
                layout=Grid(
                    groupby="run_index",
                    cols_wrap=2,
                    inner=ProgressionPlot(
                        x="discovery_eval_counts",
                        y=METRIC_COLUMNS,
                        hue="column_name",
                        linestyle="is_sanitized",
                    ),
                ),
            ),
        ],
    ),
    # Wordchain - Scatter plot test (individual points, colored by verbalization)
    "wordchain_scatter": PlotConfig(
        paths=["logs/wordchain/2025-11-19-01-04-44/"],
        quick_mode=False,
        figures=[
            Figure(
                name="Proxy vs True Reward",
                filter=lambda df: ~df.is_sanitized & df.prompt_verbalizes.notna(),
                layout=ScatterPlot(
                    x="proxy_reward",
                    y="true_reward",
                    color="prompt_verbalizes",
                    # No aggregate_by = individual points
                ),
            ),
        ],
    ),
    # Psychosis - Claude prompter
    "psychosis_claude": PlotConfig(
        paths=["logs/psychosis/2025-11-21-00-39-08/"],
        quick_mode=False,
        computed_columns={
            "prompt_type": make_prompt_type_column,
            "prompt_verbalizes": make_verbalization_float_column,
        },
        figures=[
            Figure(
                name="Final scores",
                filter=lambda df: df.prompt_type.notna(),
                layout=with_subsets(
                    BarPlot(x="prompt_type", y=METRIC_COLUMNS, hue="column_name")
                ),
            ),
            Figure(
                name="Training progression",
                # filter=lambda df: df.run_index == 0,
                layout=with_subsets(
                    ProgressionPlot(
                        x="discovery_eval_counts",
                        y=METRIC_COLUMNS,
                        hue="column_name",
                        linestyle="is_sanitized",
                    )
                ),
            ),
        ],
    ),
    # Wordchain - Multiple prompters comparison
    "wordchain_comparison": PlotConfig(
        paths=["logs/wordchain/2025-11-09-22-49-03"],
        quick_mode=True,
        figures=[
            Figure(
                name="Final scores by prompter",
                filter=lambda df: df.is_final & ~df.is_sanitized & df.subset.isna(),
                layout=BarPlot(x="prompter_name", y=METRIC_COLUMNS, hue="column_name"),
            ),
            Figure(
                name="Progression by prompter",
                filter=lambda df: ~df.is_sanitized & df.subset.isna(),
                layout=Grid(
                    groupby="prompter_name",
                    cols_wrap=2,
                    inner=ProgressionPlot(
                        x="discovery_eval_counts", y=METRIC_COLUMNS, hue="column_name"
                    ),
                ),
            ),
        ],
    ),
}

# Select which config to use
CONFIG = CONFIGS["wordchain_scatter"]

dfs = CONFIG.render_all()

In [2]:
df = dfs[1]
df = df[(df.run_index == 0) & df.prompt_verbalizes.notna()]

for _, row in df.iterrows():
    print("### VERBALIZES ###")
    print(row["prompt_verbalizes"])
    print("### CANDIDATE INDEX ###")
    print(row["candidate_index"])
    print("### INSTRUCTIONS ###")
    print(row["instructions"])
    print()

IndexError: list index out of range